# 👁️ AI VISION ASSISTANT: Huấn luyện QLoRA / SFT trên Google Colab
**Mô tả:** Notebook tự động hoá 100% để huấn luyện mô hình Vision-Language (Qwen2-VL / SmolVLM) trên GPU Tesla T4 (15GB VRAM) của Google Colab.
Notebook này kết nối trực tiếp với repository `hoang-nguyen05/AI_VISION_ASSISTANT`, tải dữ liệu thực tế từ Hugging Face Hub, huấn luyện và xuất adapter zip để tải về đặt vào thư mục `adapters/` trên máy local.

In [ ]:
# 1. Kiểm tra GPU Tesla T4 (15GB VRAM)
!nvidia-smi
import torch
print(f"CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Thiết bị GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# 2. Clone mã nguồn dự án từ GitHub và cài đặt thư viện
# Gỡ bỏ torchao cũ xung đột với PEFT trong Colab
!pip uninstall -y torchao
!rm -rf AI_VISION_ASSISTANT
!git clone https://github.com/hoang-nguyen05/AI_VISION_ASSISTANT.git
%cd AI_VISION_ASSISTANT
!pip install -q -r requirements.txt
!pip install -q huggingface_hub datasets peft bitsandbytes accelerate

In [ ]:
# 3. Tải dữ liệu thực tế OpenViVQA từ Hugging Face Hub (split dev: 165MB, ~2000 câu hỏi)
# Bạn có thể đổi sang --split train (1.34GB) nếu muốn huấn luyện toàn bộ tập dữ liệu
!python -m src.data.download --split dev
!python -m src.data.prepare_vqa
!python -m src.data.prepare_caption

In [ ]:
# 4. Khởi chạy Huấn luyện SFT LoRA trên GPU Colab T4
# Chạy huấn luyện với mô hình SmolVLM hoặc Qwen2-VL
!python -m src.training.train --config configs/smolvlm_lora.yaml

In [ ]:
# 5. Đóng gói Adapter và Tải về Máy Cục bộ
import os
from google.colab import files

!zip -r smolvlm_vietnamese_lora.zip adapters/smolvlm_vietnamese_lora
print("Bắt đầu tải adapter zip về máy...")
files.download('smolvlm_vietnamese_lora.zip')
print("Hoàn tất! Hãy giải nén file zip này và đặt vào thư mục adapters/ trên máy của bạn.")